# Topological photonics on a Haldane lattice

*em · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/em/topological_photonics)

A honeycomb array of coupled optical resonators with complex next-nearest-neighbour couplings, the photonic Haldane model. The $2\times 2$ Bloch Hamiltonian on a whole $k$-grid gives the Chern number by the Fukui–Hatsugai–Suzuki link method (einsum over link variables). A zigzag ribbon shows edge states crossing the gap, and light launched at the edge of a finite flake flows one way around a notch, while in a trivial lattice it does not get past.

**Vector form**

$$i\,\dot{\boldsymbol\psi} = H\boldsymbol\psi,\qquad H(\mathbf k) = d_0(\mathbf k)\,\mathbb 1 + \mathbf d(\mathbf k)\cdot\boldsymbol\sigma,\qquad C = \frac{1}{2\pi}\int_{\text{BZ}}\Omega\,d^2k$$

**Index (tensor) form**

$$i\dot\psi_i = H_{ij}\psi_j,\qquad H_{ab}(\mathbf k) = d_\mu(\mathbf k)\,\sigma^\mu_{ab},\qquad U_\mu(\mathbf k) = \frac{u^*_a(\mathbf k)\,u_a(\mathbf k + \hat\mu)}{\lvert\cdots\rvert},\qquad C = \frac{1}{2\pi}\sum_{\mathbf k}\arg\big[U_1(\mathbf k)\,U_2(\mathbf k + \hat 1)\,U_1^*(\mathbf k + \hat 2)\,U_2^*(\mathbf k)\big]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\psi_i$ | `psi` | (N,) | complex field amplitude in resonator $i$ of the flake |
| $H_{ij}$ | `flake_hamiltonian(pos, xi, t1, t2, phi, M, detuning)` | (N, N) | couplings $t_1$, $t_2 e^{-i\nu_{ij}\phi}$ and detuning $\pm M$ plus disorder |
| $d_\mu(\mathbf k)$ | `d_vector(k, delta, b, t1, t2, phi, M)` | (n, n, 4) | $k\cdot\delta_n$ is einsum over the coordinate $i$, for every $k$ on the grid; $\delta_n$, $b_n$ are bond vectors |
| $d_\mu\,\sigma^\mu_{ab}$ | `einsum('...m,mab->...ab', d, SIGMA)` | (n, n, 2, 2) | repeated $\mu$ ⇒ summed: the Bloch Hamiltonian at every $k$ |
| $u_a(\mathbf k)$ | `eigh(H)[1][..., 0]` | (n, n, 2) | lower-band eigenvector (its phase is arbitrary) |
| $u^*_a(\mathbf k)\,u_a(\mathbf k + \hat\mu)$ | `einsum('xya,xya->xy', u.conj(), roll(u, -1, dims=mu))` | (n, n) | repeated $a$ ⇒ summed: link variable between neighbouring $k$ points |
| $\Omega\,\Delta A$ | `plaquette_flux(U1, U2)` | (n, n) | phase of the loop of four links: Berry flux, gauge invariant |
| $-\tfrac12\varepsilon_{abc}\hat d_a\partial_x\hat d_b\partial_y\hat d_c$ | `einsum('abc,...a,...b,...c->...', EPS, dhat, ddx, ddy)` | (n, n) | repeated $a, b, c$ ⇒ summed: continuum Berry curvature, derivatives by autograd |
| $\psi_i(t)$ | `einsum('sin,stn,sn->sti', V, exp(-1j*E*t), c)` | (2, T, N) | $e^{-iHt}\psi_0$ from one eigh per lattice (topological and trivial) |

Units: the coupling $t_1 = 1$ and the resonator spacing is 1. $E$ is the detuning from the bare resonance frequency, and in a waveguide array the time $t$ becomes the propagation distance. The phase is fixed at $\phi = \pi/2$. The lattice is topological, with $C = \pm1$, when $\lvert M\rvert < 3\sqrt3\,t_2\lvert\sin\phi\rvert$; the comparison lattice uses twice that mass and has $C = 0$. The Fukui–Hatsugai–Suzuki method (J. Phys. Soc. Jpn., 2005) gives an exact integer even on a coarse grid. The bulk-boundary correspondence then promises $\lvert C\rvert$ one-way edge channels. The input light is a spot next to the bottom edge, filtered to frequencies inside the bulk gap. It runs around the notch, because there is no backward-moving state for it to scatter into. In the trivial lattice the same light just spreads into the bulk. The model is Haldane's (PRL, 1988). Haldane & Raghu (PRL, 2008) proposed its photonic version, and Wang, Chong, Joannopoulos & Soljačić (Nature, 2009) observed one-way microwave edge states in a magneto-optical photonic crystal. Rechtsman et al. (Nature, 2013) built a Floquet version from helical waveguides. Topological insulator lasers followed (Bandres et al., Science, 2018), and Ozawa et al. (Rev. Mod. Phys., 2019) review the field. More recent work, including valley-Hall photonic crystals that need no magnetic field, carries the idea to on-chip telecom and terahertz waveguides. Try raising $M$ past the critical value: $C$ drops to 0, the edge states detach from the bands, and the light no longer gets around the notch.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/em/topological_photonics/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve

T1, PHI = 1.0, math.pi / 2  # nearest-neighbour coupling (sets the units) and the Haldane phase
WIDTH, HEIGHT = 40.0, 24.0  # flake size in units of the resonator spacing


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    t2, M, R = params["t2"], params["mass"], params["defect_radius"]
    M_trivial = 2 * 3 * math.sqrt(3) * t2 * math.sin(PHI)  # twice the critical mass: a trivial gap of similar size

    # 1. topology of the infinite lattice: Chern number of the lower band (FHS on a k-grid)
    topo = solve.chern_number(T1, t2, PHI, M, 48, dev)
    triv = solve.chern_number(T1, t2, PHI, M_trivial, 48, dev)
    print(f"critical mass 3√3 t2 sin φ = {3 * math.sqrt(3) * t2:.3f}")
    for name, mass, r in [("lattice 1", M, topo), ("lattice 2", M_trivial, triv)]:
        C = round(r["C"], 4) + 0.0  # + 0.0 turns -0.0 into 0.0
        print(f"{name} (M = {mass:.2f}): Chern number C = {C:+.4f}, bulk gap {r['gap']:.3f}")
    print(f"∫Ω d²k / 2π = {topo['omega'].sum().item() * topo['cell'] / (2 * math.pi):+.4f}")

    # 2. zigzag ribbon: edge states inside the gap, and the direction they travel on the bottom edge
    bands = solve.ribbon_bands(24, T1, t2, PHI, M, 161, dev)
    bands_triv = solve.ribbon_bands(24, T1, t2, PHI, M_trivial, 161, dev)
    window = 0.3 * max(topo["gap"], 0.05)  # frequencies well inside the bulk gap
    v, k_c = solve.bottom_edge_mode(bands, window)
    print(f"bottom-edge state: k_c·√3 = {k_c * math.sqrt(3):.3f}, group velocity dE/dk = {v:+.3f}")

    # 3. finite flake with a notch cut into the bottom edge: the same edge light in both lattices
    n_times = 30 if fast_mode() else 160
    flake = solve.edge_transport(
        T1, t2, PHI, (M, M_trivial), params["disorder"], R, v, k_c, window, WIDTH, HEIGHT, n_times, dev
    )
    pos, intensity, times, transmitted = flake["pos"], flake["intensity"], flake["times"], flake["transmitted"]
    print(f"{len(pos)} resonators, the edge packet keeps {flake['kept']:.0%} of the input spot (in-gap part)")
    print(
        f"transmitted past the notch at t_max: lattice 1 {transmitted[0, -1]:.2f},  lattice 2 {transmitted[1, -1]:.2f}"
    )
    return plot.animate(
        pos, intensity, times, transmitted, topo, triv["C"], bands, bands_triv, M, M_trivial, flake["notch"], R,
        out_dir / "topological_photonics.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/em/topological_photonics/equation.py
r"""Physics → tensor translation.

Coupled-mode theory of a honeycomb lattice of identical resonators (or waveguides). The field amplitude
ψ_i in resonator i obeys a Schrödinger-like equation, with time (or propagation distance z) and
the frequency detuning playing the roles of t and E:

    i \dot ψ_i = H_{ij} ψ_j                                             (j summed)

H has three kinds of entries (the Haldane model):
    nearest-neighbour coupling        t_1
    next-nearest-neighbour coupling   t_2 e^{-i ν_{ij} φ},  ν_{ij} = ±1 for a left/right turn
    on-site detuning                  ±M on sublattice A/B

The complex t_2 breaks time-reversal symmetry (in photonics: a magneto-optical material or a
modulation). On the infinite lattice a plane wave turns H into a 2×2 Bloch Hamiltonian

    H_{ab}(k) = d_μ(k) σ^μ_{ab}                    (μ = 0…3 summed, σ^0 = identity)
    d_0 = 2 t_2 cos φ Σ_n cos(k·b_n),   d_x = t_1 Σ_n cos(k·δ_n),   d_y = -t_1 Σ_n sin(k·δ_n)
    d_z = M - 2 t_2 sin φ Σ_n sin(k·b_n)

with δ_n the three A → B bonds counted from unit cell to unit cell (δ_n - δ_1), so that H(k) is periodic in k.

The lower band's Chern number C = (1/2π) ∫ Ω d²k is an integer. Fukui–Hatsugai–Suzuki compute it on a
k-grid from the link variables U_μ(k) = ⟨u(k)|u(k + Δk_μ)⟩/|…| (the index a is summed):

    F(k) = arg[ U_1(k) U_2(k + Δk_1) U_1(k + Δk_2)^* U_2(k)^* ],      C = (1/2π) Σ_k F(k)

F/ΔA is the Berry curvature Ω on each plaquette; the gauge-invariant loop product makes C an exact integer
even on a coarse grid. The continuum formula Ω = -½ ε_{abc} d̂_a ∂_x d̂_b ∂_y d̂_c gives the same map.
"""

import math

import torch

# Pauli matrices σ^μ_{ab}, μ = 0 (identity), x, y, z: shape (4, 2, 2)
SIGMA = torch.tensor(
    [[[1, 0], [0, 1]], [[0, 1], [1, 0]], [[0, -1j], [1j, 0]], [[1, 0], [0, -1]]], dtype=torch.complex128
)
# Levi-Civita ε_{abc}: shape (3, 3, 3)
EPS = torch.zeros(3, 3, 3, dtype=torch.float64)
EPS[0, 1, 2] = EPS[1, 2, 0] = EPS[2, 0, 1] = 1.0
EPS[0, 2, 1] = EPS[2, 1, 0] = EPS[1, 0, 2] = -1.0


def d_vector(
    k: torch.Tensor, delta: torch.Tensor, b: torch.Tensor, t1: float, t2: float, phi: float, M: float
) -> torch.Tensor:
    r"""d_μ(k) for μ = 0, x, y, z.   k: (..., 2), δ_{ni}, b_{ni}: (3, 2) → (..., 4)

    k·δ_n = k_i δ_{ni} is einsum('...i,ni->...n'), then each sum over the three bonds n is .sum(-1).
    """
    kd = torch.einsum("...i,ni->...n", k, delta)  # (..., 3)
    kb = torch.einsum("...i,ni->...n", k, b)  # (..., 3)
    d0 = 2 * t2 * math.cos(phi) * torch.cos(kb).sum(-1)
    dx = t1 * torch.cos(kd).sum(-1)
    dy = -t1 * torch.sin(kd).sum(-1)
    dz = M - 2 * t2 * math.sin(phi) * torch.sin(kb).sum(-1)
    return torch.stack([d0, dx, dy, dz], dim=-1)


def bloch_hamiltonian(d: torch.Tensor) -> torch.Tensor:
    r"""H_{ab}(k) = d_μ(k) σ^μ_{ab}   (μ summed).   d: (..., 4) → (..., 2, 2) complex"""
    return torch.einsum("...m,mab->...ab", d.to(torch.complex128), SIGMA.to(d.device))


def link_variables(u: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""U_μ(k) = ⟨u(k)|u(k+Δk_μ)⟩ / |…|   (band index a summed; periodic grid).   u: (N1, N2, 2) → 2 × (N1, N2)"""
    U1 = torch.einsum("xya,xya->xy", u.conj(), torch.roll(u, -1, dims=0))
    U2 = torch.einsum("xya,xya->xy", u.conj(), torch.roll(u, -1, dims=1))
    return U1 / U1.abs(), U2 / U2.abs()


def plaquette_flux(U1: torch.Tensor, U2: torch.Tensor) -> torch.Tensor:
    r"""F(k) = arg[U_1(k) U_2(k+1̂) U_1(k+2̂)^* U_2(k)^*] ∈ (-π, π]: Berry flux through each plaquette.   → (N1, N2)"""
    loop = U1 * torch.roll(U2, -1, dims=0) * torch.roll(U1, -1, dims=1).conj() * U2.conj()
    return torch.angle(loop)


def berry_curvature_formula(dhat: torch.Tensor, ddx: torch.Tensor, ddy: torch.Tensor) -> torch.Tensor:
    r"""Ω(k) = -½ ε_{abc} d̂_a ∂_x d̂_b ∂_y d̂_c  for the lower band (a, b, c summed): minus half the solid angle of d̂.

    The sign matches the FHS fluxes above, i.e. the Berry connection A_μ = Im⟨u|∂_μ u⟩.

    dhat, ddx, ddy: (..., 3) unit vector (d_x, d_y, d_z)/|d| and its k_x, k_y derivatives → (...)
    """
    return -0.5 * torch.einsum("abc,...a,...b,...c->...", EPS.to(dhat.device), dhat, ddx, ddy)


def hoppings(
    r: torch.Tensor, r_other: torch.Tensor, xi: torch.Tensor, t1: float, t2: float, phi: float
) -> torch.Tensor:
    r"""Real-space couplings H_{ij} from site i at r_i to site j at r'_j (nearest-neighbour distance 1).

    t_1 at distance 1; t_2 e^{-i ν_{ij} φ} at distance √3, with ν_{ij} = ξ_i cos 3θ_{ij} = ±1, where θ_{ij} is the
    direction of r'_j - r_i and ξ_i = ±1 the sublattice: the same left/right-turn sign as in the Bloch d_0 and d_z.
    r: (N, 2), r_other: (N', 2), xi: (N,) → (N, N') complex
    """
    d = r_other[None, :, :] - r[:, None, :]  # (N, N', 2)
    dist = d.norm(dim=-1)
    nn = (dist - 1).abs() < 1e-6
    nnn = (dist - math.sqrt(3)).abs() < 1e-6
    nu = xi[:, None] * torch.cos(3 * torch.atan2(d[..., 1], d[..., 0])).round()  # (N, N') ±1
    H = torch.where(nn, torch.tensor(t1 + 0j, dtype=torch.complex128), torch.tensor(0j, dtype=torch.complex128))
    return torch.where(nnn, t2 * torch.exp(-1j * nu * phi), H)


def evolve(E: torch.Tensor, V: torch.Tensor, psi0: torch.Tensor, times: torch.Tensor) -> torch.Tensor:
    r"""ψ(t) = e^{-iHt} ψ_0 = V_{in} e^{-i E_n t} V^*_{jn} ψ_{0j}   (n, j summed), for a batch of lattices s.

    E: (S, N), V: (S, N, N), psi0: (N,), times: (T,) → (S, T, N)
    """
    c = torch.einsum("sjn,j->sn", V.conj(), psi0)  # (S, N) mode amplitudes
    phase = torch.exp(-1j * E[:, None, :] * times[None, :, None])  # (S, T, N)
    return torch.einsum("sin,stn,sn->sti", V, phase, c)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/em/topological_photonics/tensors.py
"""Build the concrete tensors: honeycomb vectors, the Brillouin-zone grid, a zigzag ribbon and a finite flake."""

import math

import torch

S3 = math.sqrt(3)
DELTA = torch.tensor([[0.0, 1.0], [-S3 / 2, -0.5], [S3 / 2, -0.5]], dtype=torch.float64)  # δ_{ni}: A → B bonds
B_NNN = torch.stack([DELTA[1] - DELTA[2], DELTA[2] - DELTA[0], DELTA[0] - DELTA[1]])  # b_{ni}: A → A bonds
# A → B bonds measured from cell to cell (δ_n - δ_1 are lattice vectors). With these phases H(k + G) = H(k), so
# eigenvectors can be compared across the zone boundary, as the FHS links on a periodic k-grid require.
CELL_DELTA = DELTA - DELTA[0]
A1 = torch.tensor([S3, 0.0], dtype=torch.float64)  # Bravais vectors
A2 = torch.tensor([S3 / 2, 1.5], dtype=torch.float64)


def bz_grid(n: int, device: torch.device) -> tuple[torch.Tensor, float]:
    """k = (i G_1 + j G_2)/n on an n×n periodic grid covering the Brillouin zone once: (n, n, 2), and the cell area.

    G_1, G_2 are the reciprocal vectors, a_i · G_j = 2π δ_ij.
    """
    G = 2 * math.pi * torch.linalg.inv(torch.stack([A1, A2])).T  # rows G_1, G_2
    s = torch.arange(n, dtype=torch.float64) / n
    k = s[:, None, None] * G[0] + s[None, :, None] * G[1]  # (n, n, 2)
    cell = abs(torch.linalg.det(G).item()) / n**2  # area of one k-plaquette
    return k.to(device), cell


def ribbon_cell(width: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """One period of a zigzag ribbon, infinite along x and `width` cells tall: positions (2W-2, 2), ξ = ±1 (2W-2,).

    Sites are stacked A, B, A, B, … from bottom to top. The first A and the last B would hang on a single bond
    (a "bearded" edge), so they are dropped, as in the trimmed flake: both edges are then zigzag.
    """
    n2 = torch.arange(width, dtype=torch.float64)
    a_sites = n2[:, None] * A2  # (W, 2)
    pos = torch.stack([a_sites, a_sites + DELTA[0]], dim=1).reshape(-1, 2)[1:-1]
    pos[:, 0] = torch.remainder(pos[:, 0], S3)  # fold into one period along x
    xi = torch.tensor([1.0, -1.0], dtype=torch.float64).repeat(width)[1:-1]
    return pos.to(device), xi.to(device)


def flake(width: float, height: float, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Honeycomb sites inside a width × height rectangle (zigzag edges top and bottom): positions (N, 2), ξ (N,).

    Sites left with fewer than two neighbours are trimmed, so every edge site has two couplings.
    """
    n = int(2 * (width + height))
    i, j = torch.meshgrid(torch.arange(-n, n + 1.0), torch.arange(-n, n + 1.0), indexing="ij")
    cells = i.reshape(-1, 1).double() * A1 + j.reshape(-1, 1).double() * A2  # (M, 2)
    pos = torch.cat([cells, cells + DELTA[0]])
    xi = torch.cat([torch.ones(len(cells)), -torch.ones(len(cells))]).double()
    eps = 1e-6
    keep = (pos[:, 0] > -eps) & (pos[:, 0] < width + eps) & (pos[:, 1] > -eps) & (pos[:, 1] < height + eps)
    pos, xi = _trim(pos[keep], xi[keep])
    return pos.to(device), xi.to(device)


def carve_defect(pos: torch.Tensor, xi: torch.Tensor, center: torch.Tensor, radius: float):
    """Remove every resonator within `radius` of `center` (a notch cut into the edge), then trim dangling sites."""
    keep = (pos - center).norm(dim=1) > radius
    return _trim(pos[keep], xi[keep])


def _trim(pos: torch.Tensor, xi: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Repeatedly drop sites with fewer than two nearest neighbours (distance 1)."""
    while True:
        neighbours = ((torch.cdist(pos, pos) - 1).abs() < 1e-6).sum(1)
        if (neighbours >= 2).all():
            return pos, xi
        pos, xi = pos[neighbours >= 2], xi[neighbours >= 2]


def disorder(n: int, strength: float, seed: int, device: torch.device) -> torch.Tensor:
    """Random detuning of every resonator, uniform in [-W/2, W/2]: (N,). Fixed seed."""
    g = torch.Generator().manual_seed(seed)
    return (strength * (torch.rand(n, generator=g, dtype=torch.float64) - 0.5)).to(device)


def gaussian(pos: torch.Tensor, center: torch.Tensor, width: float) -> torch.Tensor:
    """Normalised Gaussian spot of light on the resonators: (N,) complex."""
    psi = torch.exp(-((pos - center) ** 2).sum(1) / (2 * width**2)).to(torch.complex128)
    return psi / psi.norm()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/em/topological_photonics/solve.py
"""Chern number on a k-grid, zigzag-ribbon bands, and light travelling along the edge of a finite flake."""

import math

import torch

from . import tensors
from .equation import (
    berry_curvature_formula,
    bloch_hamiltonian,
    d_vector,
    evolve,
    hoppings,
    link_variables,
    plaquette_flux,
)


def chern_number(t1: float, t2: float, phi: float, M: float, n: int, device: torch.device) -> dict:
    """Fukui–Hatsugai–Suzuki on an n×n grid: lower-band eigenvectors → links → plaquette fluxes → C."""
    k, cell = tensors.bz_grid(n, device)  # (n, n, 2)
    delta, b = tensors.CELL_DELTA.to(device), tensors.B_NNN.to(device)
    H = bloch_hamiltonian(d_vector(k, delta, b, t1, t2, phi, M))  # (n, n, 2, 2): all k at once
    E, V = torch.linalg.eigh(H)  # (n, n, 2), (n, n, 2, 2)
    u = V[..., 0]  # (n, n, 2) lower band (any phase: the links are gauge invariant around a plaquette)
    F = plaquette_flux(*link_variables(u))  # (n, n)
    C = F.sum().item() / (2 * math.pi)
    gap = (E[..., 1] - E[..., 0]).min().item()
    return {"C": C, "omega": F / cell, "cell": cell, "gap": gap, "k": k}


def berry_curvature_autograd(k: torch.Tensor, t1: float, t2: float, phi: float, M: float) -> torch.Tensor:
    """Ω(k) = -½ ε_abc d̂_a ∂_x d̂_b ∂_y d̂_c, with the k-derivatives of d̂ taken by autograd.   k: (..., 2) → (...)"""
    k = k.detach().clone().requires_grad_(True)
    d = d_vector(k, tensors.CELL_DELTA.to(k.device), tensors.B_NNN.to(k.device), t1, t2, phi, M)[..., 1:]
    dhat = d / d.norm(dim=-1, keepdim=True)  # (..., 3)
    # every k is independent, so the gradient of Σ_k d̂_a(k) is ∂d̂_a/∂k at each point: (..., 2)
    grads = [torch.autograd.grad(dhat[..., a].sum(), k, retain_graph=True)[0] for a in range(3)]
    jac = torch.stack(grads, dim=-2)  # (..., 3, 2)  ∂_i d̂_a
    return berry_curvature_formula(dhat.detach(), jac[..., 0], jac[..., 1])


def ribbon_bands(width: int, t1: float, t2: float, phi: float, M: float, n_k: int, device: torch.device) -> dict:
    """Bands of a zigzag ribbon vs k_x: H(k) = Σ_m T(m) e^{i k_x m √3}, m = -1, 0, 1 neighbouring periods."""
    pos, xi = tensors.ribbon_cell(width, device)  # (2W, 2), (2W,)
    a1 = tensors.A1.to(device)
    T = torch.stack([hoppings(pos, pos + m * a1, xi, t1, t2, phi) for m in (-1, 0, 1)])  # (3, 2W, 2W)
    kx = torch.linspace(0, 2 * math.pi, n_k, dtype=torch.float64, device=device) / math.sqrt(3)  # (K,) one zone
    phase = torch.exp(1j * kx[:, None] * torch.tensor([-1.0, 0.0, 1.0], device=device) * math.sqrt(3))  # (K, 3)
    H = torch.einsum("km,mij->kij", phase, T) + torch.diag(M * xi).to(torch.complex128)  # (K, 2W, 2W)
    E, V = torch.linalg.eigh(H)  # (K, 2W), (K, 2W, 2W)
    weight = V.abs() ** 2  # (K, site, band)
    rows = len(pos) // 4  # bottom and top quarter of the ribbon
    bottom = weight[:, :rows].sum(1)  # (K, band)
    top = weight[:, -rows:].sum(1)
    return {"kx": kx, "E": E, "bottom": bottom, "top": top}


def bottom_edge_mode(bands: dict, window: float) -> tuple[float, float]:
    """Group velocity v = dE/dk and momentum k_c of the bottom-edge state inside the gap (|E| < window).

    A straight-line fit E ≈ v (k - k_c) through the states that live on the bottom edge; the sign of v is the
    direction light travels there. Returns (0, 0) when there is no such state (a trivial gap).
    """
    kx, E, bottom = bands["kx"], bands["E"], bands["bottom"]
    on_edge = (bottom > 0.6) & (E.abs() < window)  # (K, band)
    if on_edge.sum() < 3:
        return 0.0, 0.0
    k = kx[:, None].expand_as(E)[on_edge]
    e = E[on_edge]
    v = (((k - k.mean()) * (e - e.mean())).sum() / ((k - k.mean()) ** 2).sum()).item()
    return v, (k.mean() - e.mean() / v).item()


def flake_hamiltonian(pos, xi, t1: float, t2: float, phi: float, M: float, detuning: torch.Tensor) -> torch.Tensor:
    """H_ij of the finite flake: couplings plus on-site M ξ_i and a random detuning. (N, N) complex Hermitian."""
    return hoppings(pos, pos, xi, t1, t2, phi) + torch.diag(M * xi + detuning).to(torch.complex128)


def edge_packet(
    H: torch.Tensor, pos: torch.Tensor, center: torch.Tensor, k_c: float, window: float
) -> tuple[torch.Tensor, float]:
    """A spot of light near the edge with the edge mode's momentum e^{i k_c x}, filtered to frequencies
    |E| < window (inside the bulk gap). Returns ψ_0 (N,) and the fraction of the spot that was kept."""
    spot = tensors.gaussian(pos, center, 1.5) * torch.exp(1j * k_c * pos[:, 0])
    E, V = torch.linalg.eigh(H)
    c = V.conj().T @ spot  # mode amplitudes
    c = torch.where(E.abs() < window, c, torch.zeros_like(c))
    kept = (c.abs() ** 2).sum().item()
    if kept < 1e-8:  # no states in the window (a trivial gap): fall back to the bare spot
        return spot, 0.0
    psi = V @ c
    return psi / psi.norm(), kept


def detector(pos, notch, R: float, direction: float, width: float, height: float, strip: float = 3.0) -> torch.Tensor:
    """Resonators downstream of the notch and within `strip` of the flake's outer boundary: boolean (N,)."""
    x, y = pos[:, 0], pos[:, 1]
    to_boundary = torch.stack([x, width - x, y - y.min(), height - y]).min(0).values
    return (direction * (x - notch[0]) > R + 1) & (to_boundary < strip)


def propagate(Hs: torch.Tensor, psi0: torch.Tensor, times: torch.Tensor) -> torch.Tensor:
    """|ψ_i(t)|² for each lattice in the batch, ψ(t) = e^{-iHt} ψ_0 via one eigh each.   Hs: (S, N, N) → (S, T, N)"""
    E, V = torch.linalg.eigh(Hs)
    return evolve(E, V, psi0, times).abs() ** 2


def edge_transport(
    t1: float, t2: float, phi: float, masses: tuple, disorder: float, R: float, v: float, k_c: float, window: float,
    width: float, height: float, n_times: int, device: torch.device,
) -> dict:  # fmt: skip
    """Launch edge light upstream of a notch in a disordered flake and follow it in one lattice per mass.

    Returns positions, intensity |ψ|² (S, T, N), times (T,), the fraction past the notch along the edge (S, T),
    the notch centre and the fraction of the input spot that lay inside the gap.
    """
    pos, xi = tensors.flake(width, height, device)
    notch = torch.tensor([width / 2, 0.0], dtype=torch.float64, device=device)
    pos, xi = tensors.carve_defect(pos, xi, notch, R)
    detuning = tensors.disorder(len(pos), disorder, seed=1, device=device)
    Hs = torch.stack([flake_hamiltonian(pos, xi, t1, t2, phi, m, detuning) for m in masses])  # (S, N, N)

    s = 1.0 if v >= 0 else -1.0  # direction of travel along the bottom edge
    source = torch.tensor([width / 2 - s * width / 4, 1.0], dtype=torch.float64, device=device)
    psi0, kept = edge_packet(Hs[0], pos, source, k_c, window)
    t_max = (0.5 * width + math.pi * R + 0.3 * height) / max(abs(v), 0.2)  # long enough to pass the notch
    times = torch.linspace(0, t_max, n_times, dtype=torch.float64, device=device)
    intensity = propagate(Hs, psi0, times)  # (S, T, N)
    transmitted = intensity[:, :, detector(pos, notch, R, s, width, height)].sum(-1)  # (S, T)
    return {
        "pos": pos,
        "intensity": intensity,
        "times": times,
        "transmitted": transmitted,
        "notch": notch,
        "kept": kept,
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/em/topological_photonics/plot.py
"""Top: light intensity |ψ|² on the topological and the trivial flake. Bottom: Berry curvature, ribbon bands
with edge states, and the fraction of light that has passed the notch."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import LinearSegmentedColormap

from problems._common.plotting import frame_indices, save_gif

LIGHT = LinearSegmentedColormap.from_list("light", ["#e5e7eb", "#60a5fa", "#1e3a8a"], N=8)  # discrete: small GIF


def animate(
    pos: torch.Tensor, intensity: torch.Tensor, times: torch.Tensor, transmitted: torch.Tensor, topo: dict,
    C_trivial: float, bands: dict, bands_triv: dict, M: float, M_trivial: float, notch: torch.Tensor, R: float,
    path: Path,
) -> Path:  # fmt: skip
    xy, inten, t, tr = pos.cpu().numpy(), intensity.cpu().numpy(), times.cpu().numpy(), transmitted.cpu().numpy()
    peak = 0.4 * inten[0, 0].max()  # colour scale fixed by the input light

    fig = plt.figure(figsize=(7.6, 5.4))
    grid = fig.add_gridspec(2, 3, height_ratios=[1.15, 1])
    top = grid[0, :].subgridspec(1, 2, wspace=0.05)
    names = [f"M = {M:.2f}:  C = {round(topo['C']):+d}", f"M = {M_trivial:.2f}:  C = {round(C_trivial):+d} (trivial)"]
    dots, flakes = [], []
    for s in range(2):
        ax = fig.add_subplot(top[0, s])
        dots.append(ax.scatter(xy[:, 0], xy[:, 1], s=9, c=inten[s, 0], cmap=LIGHT, vmin=0, vmax=peak, lw=0))
        ax.add_patch(plt.Circle(notch.cpu().numpy(), R, fill=False, ls="--", color="#9ca3af", lw=1))
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        ax.set_title(names[s], fontsize=9)
        flakes.append(ax)
    clock = flakes[0].text(0.01, 0.98, "", transform=flakes[0].transAxes, va="top", fontsize=8)

    # Berry curvature of the lower band on the k-grid (reduced coordinates k_1, k_2 ∈ [0, 1))
    ax = fig.add_subplot(grid[1, 0])
    om = topo["omega"].cpu().numpy()
    lim = max(np.abs(om).max(), 1e-12)
    ax.imshow(om.T, origin="lower", extent=(0, 1, 0, 1), cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_xlabel("k₁ / |G₁|")
    ax.set_ylabel("k₂ / |G₂|")
    ax.grid(False)
    ax.set_title(f"Berry curvature Ω,  C = {topo['C']:+.2f}", fontsize=8)

    # ribbon bands: light grey = trivial lattice; dark = topological, with red/blue = bottom/top edge states
    ax = fig.add_subplot(grid[1, 1])
    kx = bands["kx"].cpu().numpy() * np.sqrt(3)
    E, bottom, top_w = bands["E"].cpu().numpy(), bands["bottom"].cpu().numpy(), bands["top"].cpu().numpy()
    ax.plot(kx, bands_triv["E"].cpu().numpy(), color="#d1d5db", lw=0.6)
    ax.plot(kx, E, color="#4b5563", lw=0.5)
    kk = np.repeat(kx[:, None], E.shape[1], 1)
    ax.plot(kk[bottom > 0.6], E[bottom > 0.6], ".", color="#dc2626", ms=2)
    ax.plot(kk[top_w > 0.6], E[top_w > 0.6], ".", color="#2563eb", ms=2)
    ax.set_xlim(0, 2 * np.pi)
    ax.set_xticks([0, np.pi, 2 * np.pi], ["0", "π", "2π"])
    ax.set_ylim(-1.6, 1.6)
    ax.set_xlabel("k_x a√3")
    ax.set_ylabel("detuning E")
    ax.set_title("ribbon: bottom / top edge", fontsize=8)

    # fraction of the light that has passed the notch and still runs along the edge
    ax = fig.add_subplot(grid[1, 2])
    ax.plot(t, tr[0], color="#2563eb", lw=1.8, label="topological")
    ax.plot(t, tr[1], color="#9ca3af", lw=1.8, label="trivial")
    ax.set_xlim(0, t[-1])
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("time t")
    ax.set_ylabel("edge light past notch")
    ax.legend(fontsize=7, loc="upper left")
    cursor = ax.axvline(0, color="#f59e0b", lw=1.2)

    def update(step: int):
        for s in range(2):
            dots[s].set_array(inten[s, step])
        cursor.set_xdata([t[step]])
        clock.set_text(f"t = {t[step]:.1f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 60), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    't2': 0.15,
    'mass': 0.0,
    'disorder': 0.3,
    'defect_radius': 3.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.em.topological_photonics")]:
    del sys.modules[name]
main = importlib.import_module("problems.em.topological_photonics.main")
problem_dir = Path("problems/em/topological_photonics")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
